# Task 3.2 — Gate Utilization Summary

Per `gate + operational_date`:
- `scheduled_flights`, `actual_flights`, `total_occupied_mins`, `total_idle_mins`
- `avg_turn_time_mins`, `turn_time_vs_minimum_ratio` (actual / SLA minimum)
- `utilization_pct`: occupied_mins / total_available_mins_per_day × 100
- Alert tier: **OVER_UTILIZED** if > 90%, **UNDER_UTILIZED** if < 40%
- Used by airport commercial team to rebalance gate assignments quarterly


In [ ]:
# ── Imports & Configuration ────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import *
from delta.tables import DeltaTable

BRONZE_DIR = "/FileStore/delta_lake/bronze"
SILVER_DIR = "/FileStore/delta_lake/silver"
GOLD_DIR = "/FileStore/delta_lake/gold"

try:
    spark
except NameError:
    spark = (SparkSession.builder
        .appName("Task_3_2_Gate_Utilization")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog",
                "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate())


## Step 1 — Read Silver Flight Operations

In [ ]:
# ── Read flight operations ────────────────────────────────────────
flight_ops_df = spark.read.format("delta").load(f"{SILVER_DIR}/flight_operations")

# Filter to flights that have gate event data
gate_flights_df = flight_ops_df.filter(
    F.col("gate_id").isNotNull() & F.col("gate_open_ts").isNotNull()
)

print(f"Flights with gate data: {gate_flights_df.count()}")
gate_flights_df.select(
    "flight_id", "gate_id", "gate_open_ts", "gate_released_ts",
    "gate_turn_time_mins", "operational_date"
).show(5, truncate=False)


## Step 2 — Compute Per-Gate Utilization Metrics

In [ ]:
# ── Per-gate utilization metrics ──────────────────────────────────
AVAILABLE_MINS_PER_DAY = 18 * 60  # 18 hours operational (06:00–00:00)

# Compute occupied time per flight at gate
gate_flights_df = gate_flights_df.withColumn(
    "occupied_mins",
    F.round(
        (F.unix_timestamp(F.coalesce("gate_released_ts", "gate_closed_ts", "pushback_ts")) -
         F.unix_timestamp("gate_open_ts")) / 60, 1
    )
)

# Aggregate per gate per day
gate_util_df = (gate_flights_df
    .groupBy("gate_id", "terminal", "operational_date")
    .agg(
        F.count("*").alias("total_flights"),
        F.sum(F.when(F.col("status") != "CNX", 1).otherwise(0))
            .alias("actual_flights"),
        F.round(F.sum("occupied_mins"), 1).alias("total_occupied_mins"),
        F.round(F.avg("gate_turn_time_mins"), 1).alias("avg_turn_time_mins"),
        F.round(F.min("gate_turn_time_mins"), 1).alias("min_turn_time_mins"),
        F.round(F.max("gate_turn_time_mins"), 1).alias("max_turn_time_mins")
    )
)

# Compute idle time and utilization pct
gate_util_df = (gate_util_df
    .withColumn("total_available_mins", F.lit(AVAILABLE_MINS_PER_DAY))
    .withColumn("total_idle_mins",
        F.round(F.lit(AVAILABLE_MINS_PER_DAY) - F.col("total_occupied_mins"), 1))
    .withColumn("utilization_pct",
        F.round(F.col("total_occupied_mins") / F.lit(AVAILABLE_MINS_PER_DAY) * 100, 1))
)

print(f"Gate-day records: {gate_util_df.count()}")
gate_util_df.show(5, truncate=False)


## Step 3 — Alert Tier Classification

In [ ]:
# ── Alert tier classification ─────────────────────────────────────
gate_util_df = gate_util_df.withColumn(
    "alert_tier",
    F.when(F.col("utilization_pct") > 90, "OVER_UTILIZED")
     .when(F.col("utilization_pct") < 40, "UNDER_UTILIZED")
     .otherwise("OPTIMAL")
)

gate_util_df = gate_util_df.withColumn("processing_ts", F.current_timestamp())

print("\nAlert Tier Distribution:")
gate_util_df.groupBy("alert_tier").count().orderBy("count", ascending=False).show()

print("\nUtilization by Terminal:")
gate_util_df.groupBy("terminal").agg(
    F.round(F.avg("utilization_pct"), 1).alias("avg_utilization_pct"),
    F.count("*").alias("gates")
).show()


## Step 4 — Write to Gold Delta

In [ ]:
# ── Write gold.gate_utilization_summary ────────────────────────────
gold_gate_path = f"{GOLD_DIR}/gate_utilization_summary"

(gate_util_df.write
    .format("delta")
    .mode("overwrite")
    .save(gold_gate_path))

print(f"Written to {gold_gate_path}")
result_df = spark.read.format("delta").load(gold_gate_path)
print(f"Gold gate_utilization_summary total: {result_df.count()}")
result_df.orderBy("utilization_pct", ascending=False).show(10, truncate=False)
